## Регрессия для IC50

In [2]:
!pip install catboost -q

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

In [3]:
data = pd.read_csv('data/preprocessed_data.csv')
data.head()

,NumSaturatedHeterocycles,VSA_EState4,EState_VSA8,Kappa1,SlogP_VSA3,fr_benzene,BCUT2D_CHGLO,MaxEStateIndex,NumAliphaticHeterocycles,NumRotatableBonds,...,log_IC50,log_CC50,log_SI,"IC50, mM","CC50, mM",SI,IC50_median,CC50_median,SI_median,SI_8
0,0,4.807589,41.542423,20.606247,0.000000,0,-2.343082,5.094096,0,7,...,0.795141,2.244234,1.449093,6.239374,175.482382,28.125000,0,0,1,1
1,0,2.153503,52.176000,21.163454,0.000000,0,-2.394690,3.961417,0,9,...,-0.112478,0.732620,0.845098,0.771831,5.402819,7.000000,0,0,1,0
2,0,2.184127,69.733111,25.026112,0.000000,0,-2.477203,2.627117,0,9,...,2.349877,2.207210,-0.142668,223.808778,161.142320,0.720000,1,0,0,0
3,0,4.827852,41.542423,21.567454,0.000000,0,-2.342885,5.097360,0,8,...,0.231883,2.032843,1.800960,1.705624,107.855654,63.235294,0,0,1,1
4,0,9.071783,90.073360,23.194917,6.420822,2,-2.342009,5.150510,0,4,...,2.029917,2.143861,0.113943,107.131532,139.270991,1.300000,1,0,0,0


In [4]:
# Подготовим данные
ex_colums = ['log_IC50', 'log_CC50', 'log_SI', 'IC50, mM', 'CC50, mM', 'SI',
                'IC50_median', 'CC50_median', 'SI_median', 'SI_8']

X = data.drop(columns=ex_colums)
y = data['log_IC50']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          
    random_state=RANDOM_STATE
)
X_train.shape

(798, 30)

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipline = Pipeline([('scaler', 'passthrough'), ('models', LinearRegression())])

In [6]:
# Параметры для различных моделей
param_grid = [
    {
        'scaler': [StandardScaler(), MinMaxScaler(), 'passthrough'],
        'models': [LinearRegression()]
    },
    {
        'scaler': ['passthrough'],
        'models': [RandomForestRegressor(random_state=RANDOM_STATE)],
        'models__n_estimators': [100, 150, 300],
        'models__max_depth': [None, 30, 40],
        'models__min_samples_split': [2, 5, 10],
        'models__min_samples_leaf': [1, 3, 5]
    },
    {
       'scaler': ['passthrough'],
       'models': [CatBoostRegressor(random_state=RANDOM_STATE, verbose=0)],
       'models__iterations': [100, 200, 500, 1000],
       'models__learning_rate': [0.01, 0.1],
       'models__depth': [5, 10, 15]
    }
]

In [7]:
# Для подбора гиперпараметров воспользуемся RandomizedSearchCV
random_search = RandomizedSearchCV(
    pipline,
    param_distributions=param_grid,
    n_iter=15,
    cv=cv,
    scoring='neg_mean_squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [8]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scaler', 'passthrough'),
                                             ('models', LinearRegression())]),
                   n_iter=15, n_jobs=-1,
                   param_distributions=[{'models': [LinearRegression()],
                                         'scaler': [StandardScaler(),
                                                    MinMaxScaler(),
                                                    'passthrough']},
                                        {'models': [RandomForestRegressor(random_state=42)],
                                         'models__max_dept...
                                         'models__min_samples_split': [2, 5,
                                                                       10],
                                         'models__n_estimators': [100, 150,
                                                                  300],
                                         'scaler': ['passthrough']},
                                        {'models': [CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)],
                                         'models__depth': [5, 10, 15],
                                         'models__iterations': [100, 200, 500,
                                                                1000],
                                         'models__learning_rate': [0.01, 0.1],
                                         'scaler': ['passthrough']}],
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=1)

In [9]:
print('Лучшая модель и её параметры:\n\n', random_search.best_params_) 
print('Метрика  для лучшей модели:\n', random_search.best_score_) 

Лучшая модель и её параметры:

 {'scaler': 'passthrough', 'models__learning_rate': 0.1, 'models__iterations': 200, 'models__depth': 10, 'models': CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)}
Метрика  для лучшей модели:
 -0.46328639022911544


In [10]:
best_model = random_search.best_estimator_

# Оценим на тестовой выборке, посмотрим на несколько метрик
y_pred_log = best_model.predict(X_test)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae_log = mean_absolute_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)

print(f'rmse_log: {rmse_log}')
print(f'mae_log: {mae_log}')
print(f'r2_log: {r2_log}')

rmse_log: 0.6847271012343155
mae_log: 0.5212951507564805
r2_log: 0.5442398052733829


In [ ]:
# Посмотрим на метрики без логарифма
y_pred = 10 ** y_pred_log
y_test_ic50 = data.loc[y_test.index, 'IC50, mM']
rmse = np.sqrt(mean_squared_error(y_test_ic50, y_pred))
mae = mean_absolute_error(y_test_ic50, y_pred)
r2 = r2_score(y_test_ic50, y_pred)

print(f'rmse: {rmse}')
print(f'mae: {mae}')
print(f'r2: {r2}')

rmse: 407.52573807239304
mae: 188.083204512625
r2: 0.39876048055365754


In [13]:
data['IC50, mM'].describe()

count     998.000000
mean      221.118757
std       400.510657
min         0.003517
25%        12.491340
50%        45.992006
75%       224.408630
max      4128.529377
Name: IC50, mM, dtype: float64

In [15]:
# Посмторим насколько лучше полученная модель, чем просто заполнение средним
y_pred_log_mean = np.full_like(y_test, y_train.mean())
rmse_log_mean = np.sqrt(mean_squared_error(y_test, y_pred_log_mean))
print(f'RMSE по среднему: {rmse_log_mean}')

RMSE по среднему: 1.014447173502286


In [22]:
# Сравнение метрик моделей
res = pd.DataFrame(random_search.cv_results_)
res['model'] = res['params'].apply(lambda x: type(x['models']).__name__)
res['rmse'] = np.sqrt(-res['mean_test_score'])
res.groupby('model')['rmse'].agg(['mean', 'min', 'max'])

,mean,min,max
model,,,
CatBoostRegressor,0.680651,0.680651,0.680651
LinearRegression,0.799575,0.799575,0.799575
RandomForestRegressor,0.691878,0.689044,0.695595


В ходе выполнения задачи регресии для IC50 были расмотрены несколько моделей с разными гиперпараметрами (LinearRegression()RandomForestRegressor(), CatBoostRegressor()). Используя RandomizedSearchCV следующая модель была определена лучшей: {'scaler': 'passthrough', 'models__learning_rate': 0.1, 'models__iterations': 200, 'models__depth': 10, 'models': CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)}. 

CatBoostRegressor показал наилучшее результаты, так как он лучше может улавливать нелинейные зависимости, RandomForestRegressor очень близок по результатам, но чуть-чуть уступает, LinearRegression справилась значительно хуже, но это ожидаемо, так как максмальная корреляция с целевой переменной не превшала 0.3.

Метрика в логарифмической шкале:

rmse_log: 0.6847271012343155
mae_log: 0.5212951507564805
r2_log: 0.5442398052733829

Модель объясняет 54% дисперсии логарифма IC50.

Метрика в исходной шкале:

rmse: 407.52573807239304
mae: 188.083204512625
r2: 0.39876048055365754

Относительно высокие абсолютные ошибки получаются из-за разброса переменной от 0.003517 до 4128.529377 mM. Также, сравнив полученную модель с простым заполнением средним, видно, что ошибка у CatBoostRegressor значительно меньше - модель показывает значимо лучшие результаты, чем простое предсказание средним.

В качестве рекомендации по улучшение можно попробовать применить PCA для снижения размерности, попробовать использовать нейронные сети в качестве модели.